## Experiment: Spark Data Cleaning, Transformation and Aggregation using DataFrames
Objective

Understand Spark fundamentals.

Learn DataFrame concepts and immutability.

Perform data cleaning.

Apply filtering conditions.

Use aggregation functions.

Group data using groupBy().

Modify schema.

Understand wide transformations and shuffle operations.

Build a complete data processing pipeline.

## Step 1: Install PySpark

Install PySpark library in Google Colab.

In [6]:
!pip install pyspark

## Step 2: Import Libraries

Import SparkSession and SQL functions required for DataFrame operations.

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

## Step 3: Create Spark Session

Spark Session acts as the entry point for Spark applications.

In [8]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("SparkAssignment") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


## Step 4: Create Sample Dataset

Create sample sales data because no dataset is provided.

In [9]:
# Sample dataset
data = [
    (1, "Priya", 22, "Electronics", "North", 5000),
    (2, "Rahul", 25, "Clothing", "South", 3000),
    (3, "Amit", None, "Electronics", "North", 7000),
    (4, "Sneha", 28, "Furniture", "East", 4000),
    (5, "Karan", 35, "Clothing", "West", None),
    (6, "Priya", 22, "Electronics", "North", 5000),  # Duplicate record
    (7, "Tanvi", 30, "Furniture", "East", 6000),
    (8, "Neha", 19, None, "South", 2000),
    (9, "Rohit", 40, "Electronics", "West", 8000)
]

# Define column names
columns = ["ID", "Name", "Age", "Category", "Region", "Sales"]

## Step 5: Create DataFrame

Convert raw data into a Spark DataFrame.

In [10]:
# Create DataFrame
df = spark.createDataFrame(data, columns)

# Display DataFrame
df.show()

+---+-----+----+-----------+------+-----+
| ID| Name| Age|   Category|Region|Sales|
+---+-----+----+-----------+------+-----+
|  1|Priya|  22|Electronics| North| 5000|
|  2|Rahul|  25|   Clothing| South| 3000|
|  3| Amit|NULL|Electronics| North| 7000|
|  4|Sneha|  28|  Furniture|  East| 4000|
|  5|Karan|  35|   Clothing|  West| NULL|
|  6|Priya|  22|Electronics| North| 5000|
|  7|Tanvi|  30|  Furniture|  East| 6000|
|  8| Neha|  19|       NULL| South| 2000|
|  9|Rohit|  40|Electronics|  West| 8000|
+---+-----+----+-----------+------+-----+



## Step 6: Check Schema

Display column names and data types.


In [11]:
# Print DataFrame schema
df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Sales: long (nullable = true)



## Step 7: Remove Duplicates

Remove duplicate records from the dataset.

In [12]:
# Remove duplicate rows
df_no_duplicates = df.dropDuplicates()

# Display cleaned data
df_no_duplicates.show()

+---+-----+----+-----------+------+-----+
| ID| Name| Age|   Category|Region|Sales|
+---+-----+----+-----------+------+-----+
|  1|Priya|  22|Electronics| North| 5000|
|  2|Rahul|  25|   Clothing| South| 3000|
|  4|Sneha|  28|  Furniture|  East| 4000|
|  6|Priya|  22|Electronics| North| 5000|
|  7|Tanvi|  30|  Furniture|  East| 6000|
|  9|Rohit|  40|Electronics|  West| 8000|
|  3| Amit|NULL|Electronics| North| 7000|
|  5|Karan|  35|   Clothing|  West| NULL|
|  8| Neha|  19|       NULL| South| 2000|
+---+-----+----+-----------+------+-----+



## Step 8: Find Null Values

Count null values present in each column.

In [13]:
# Count null values column-wise
df_no_duplicates.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_no_duplicates.columns
]).show()

+---+----+---+--------+------+-----+
| ID|Name|Age|Category|Region|Sales|
+---+----+---+--------+------+-----+
|  0|   0|  1|       1|     0|    1|
+---+----+---+--------+------+-----+



## Step 9: Handle Null Values


Replace missing values with default values.

In [14]:
# Replace null values
df_clean = df_no_duplicates.fillna({
    "Age": 0,
    "Category": "Unknown",
    "Sales": 0
})

# Display updated data
df_clean.show()

+---+-----+---+-----------+------+-----+
| ID| Name|Age|   Category|Region|Sales|
+---+-----+---+-----------+------+-----+
|  1|Priya| 22|Electronics| North| 5000|
|  2|Rahul| 25|   Clothing| South| 3000|
|  4|Sneha| 28|  Furniture|  East| 4000|
|  6|Priya| 22|Electronics| North| 5000|
|  7|Tanvi| 30|  Furniture|  East| 6000|
|  9|Rohit| 40|Electronics|  West| 8000|
|  3| Amit|  0|Electronics| North| 7000|
|  5|Karan| 35|   Clothing|  West|    0|
|  8| Neha| 19|    Unknown| South| 2000|
+---+-----+---+-----------+------+-----+



## Step 10: Handle Empty Names

Replace blank names with "Unknown".

In [15]:
# Replace blank names
df_clean = df_clean.withColumn(
    "Name",
    when(col("Name")=="","Unknown")
    .otherwise(col("Name"))
)

df_clean.show()

+---+-----+---+-----------+------+-----+
| ID| Name|Age|   Category|Region|Sales|
+---+-----+---+-----------+------+-----+
|  1|Priya| 22|Electronics| North| 5000|
|  2|Rahul| 25|   Clothing| South| 3000|
|  4|Sneha| 28|  Furniture|  East| 4000|
|  6|Priya| 22|Electronics| North| 5000|
|  7|Tanvi| 30|  Furniture|  East| 6000|
|  9|Rohit| 40|Electronics|  West| 8000|
|  3| Amit|  0|Electronics| North| 7000|
|  5|Karan| 35|   Clothing|  West|    0|
|  8| Neha| 19|    Unknown| South| 2000|
+---+-----+---+-----------+------+-----+



## Step 11: Filter Age Between 20 and 35

Select customers aged between 20 and 35

In [16]:
# Filter age range
df_clean.filter(
    (col("Age")>=20) &
    (col("Age")<=35)
).show()

+---+-----+---+-----------+------+-----+
| ID| Name|Age|   Category|Region|Sales|
+---+-----+---+-----------+------+-----+
|  1|Priya| 22|Electronics| North| 5000|
|  2|Rahul| 25|   Clothing| South| 3000|
|  4|Sneha| 28|  Furniture|  East| 4000|
|  6|Priya| 22|Electronics| North| 5000|
|  7|Tanvi| 30|  Furniture|  East| 6000|
|  5|Karan| 35|   Clothing|  West|    0|
+---+-----+---+-----------+------+-----+



## Step 12: Filter Electronics Category

Select Electronics products only.

In [17]:
# Electronics category
df_clean.filter(
    col("Category")=="Electronics"
).show()

+---+-----+---+-----------+------+-----+
| ID| Name|Age|   Category|Region|Sales|
+---+-----+---+-----------+------+-----+
|  1|Priya| 22|Electronics| North| 5000|
|  6|Priya| 22|Electronics| North| 5000|
|  9|Rohit| 40|Electronics|  West| 8000|
|  3| Amit|  0|Electronics| North| 7000|
+---+-----+---+-----------+------+-----+



## Step 13: Filter North Region

Select records from North region.

In [18]:
# North region
df_clean.filter(
    col("Region")=="North"
).show()

+---+-----+---+-----------+------+-----+
| ID| Name|Age|   Category|Region|Sales|
+---+-----+---+-----------+------+-----+
|  1|Priya| 22|Electronics| North| 5000|
|  6|Priya| 22|Electronics| North| 5000|
|  3| Amit|  0|Electronics| North| 7000|
+---+-----+---+-----------+------+-----+



## Step 14: Aggregation Functions

Calculate summary statistics.

In [19]:
# Perform aggregation
df_clean.select(
    count("Sales").alias("Total_Records"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales")
).show()

+-------------+-----------+-----------------+-------------+-------------+
|Total_Records|Total_Sales|    Average_Sales|Minimum_Sales|Maximum_Sales|
+-------------+-----------+-----------------+-------------+-------------+
|            9|      40000|4444.444444444444|            0|         8000|
+-------------+-----------+-----------------+-------------+-------------+



## Step 15: Group By Category

Calculate total sales for each category.

In [20]:
# Group by category
category_sales = df_clean.groupBy("Category") \
                         .agg(sum("Sales").alias("Total_Sales"))

category_sales.show()

+-----------+-----------+
|   Category|Total_Sales|
+-----------+-----------+
|Electronics|      25000|
|   Clothing|       3000|
|    Unknown|       2000|
|  Furniture|      10000|
+-----------+-----------+



## Step 16: Apply Condition on Aggregated Data
Show categories having sales greater than 5000

In [21]:
# Filter aggregated result
category_sales.filter(
    col("Total_Sales") > 5000
).show()


+-----------+-----------+
|   Category|Total_Sales|
+-----------+-----------+
|Electronics|      25000|
|  Furniture|      10000|
+-----------+-----------+



## Step 17: Rename Column

Change Sales column name to Revenue.

In [22]:
# Rename column
df_new = df_clean.withColumnRenamed(
    "Sales",
    "Revenue"
)

df_new.show()

+---+-----+---+-----------+------+-------+
| ID| Name|Age|   Category|Region|Revenue|
+---+-----+---+-----------+------+-------+
|  1|Priya| 22|Electronics| North|   5000|
|  2|Rahul| 25|   Clothing| South|   3000|
|  4|Sneha| 28|  Furniture|  East|   4000|
|  6|Priya| 22|Electronics| North|   5000|
|  7|Tanvi| 30|  Furniture|  East|   6000|
|  9|Rohit| 40|Electronics|  West|   8000|
|  3| Amit|  0|Electronics| North|   7000|
|  5|Karan| 35|   Clothing|  West|      0|
|  8| Neha| 19|    Unknown| South|   2000|
+---+-----+---+-----------+------+-------+



## Step 18: Change Data Type

Convert Revenue column to Integer type.

In [23]:
from pyspark.sql.types import IntegerType

# Cast datatype
df_new = df_new.withColumn(
    "Revenue",
    col("Revenue").cast(IntegerType())
)

df_new.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = false)
 |-- Category: string (nullable = false)
 |-- Region: string (nullable = true)
 |-- Revenue: integer (nullable = false)



## Step 19: Wide Transformation (Shuffle)

groupBy() is a wide transformation because data moves across partitions.

In [24]:
# Wide transformation example
df_clean.groupBy("Category") \
        .sum("Sales") \
        .show()

+-----------+----------+
|   Category|sum(Sales)|
+-----------+----------+
|Electronics|     25000|
|   Clothing|      3000|
|    Unknown|      2000|
|  Furniture|     10000|
+-----------+----------+



## Step 20: Complete Data Processing Pipeline

Combine cleaning, filtering and aggregation into one pipeline.

In [25]:
# Complete ETL pipeline
final_result = (
    df
    .dropDuplicates()
    .fillna({
        "Age":0,
        "Category":"Unknown",
        "Sales":0
    })
    .withColumn(
        "Name",
        when(col("Name")=="","Unknown")
        .otherwise(col("Name"))
    )
    .filter(col("Age") >= 20)
    .groupBy("Category")
    .agg(
        count("*").alias("Customer_Count"),
        sum("Sales").alias("Total_Sales"),
        avg("Sales").alias("Average_Sales")
    )
)

final_result.show()

+-----------+--------------+-----------+-------------+
|   Category|Customer_Count|Total_Sales|Average_Sales|
+-----------+--------------+-----------+-------------+
|Electronics|             3|      18000|       6000.0|
|   Clothing|             2|       3000|       1500.0|
|  Furniture|             2|      10000|       5000.0|
+-----------+--------------+-----------+-------------+

